# CloudCost IQ — MVP 2 model training (Google Colab)

This notebook trains the **global multi-horizon XGBoost quantile** model (P10/P50/P90, 1–30 day direct horizons) using **reusable `app.ml.*` code** so training and inference share the same feature definitions.

**Data policy**
- **Production claims require real normalized billing exports** (AWS/Azure/GCP/FOCUS) with sufficient history (180+ days, 365+ preferred), one currency and one cost basis.
- **Synthetic/demo data** is for pipeline demonstration and automated tests only and must never be used to claim production accuracy. This notebook can optionally generate a clearly labelled synthetic dataset for an end-to-end dry run.

**Before running**
1. Set runtime to GPU if desired (CPU also supported; `tree_method=hist`).
2. In **Section 2**, set your repo URL/ref (or upload the project zip).
3. In **Section 3**, choose a data source: `drive`, `upload`, or `synthetic_demo` (demo only).


## Section 1 — Environment setup

In [ ]:
import os, platform, random, subprocess, sys
from pathlib import Path

SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print('Python', platform.python_version())
print('Platform', platform.platform())

# Core ML deps (safe to run before cloning; project requirements are installed in Section 2)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'xgboost>=2.0,<3', 'scikit-learn>=1.4,<2', 'pandas>=2.2', 'numpy>=1.26',
    'pyarrow>=15', 'pydantic>=2.7', 'pydantic-settings>=2.3', 'PyYAML>=6',
    'plotly>=5.22', 'mlflow>=2.12,<4', 'shap>=0.45', 'joblib>=1.3,<2'])

import numpy as np
import pandas as pd
import sklearn
import xgboost
import pyarrow

np.random.seed(SEED)
print('xgboost', xgboost.__version__)
print('pandas', pd.__version__)
print('numpy', np.__version__)
print('scikit-learn', sklearn.__version__)
print('pyarrow', pyarrow.__version__)


## Section 2 — Load the project (reproducible checkout)

In [ ]:
# Configure one of: (A) clone from Git, or (B) use an uploaded/existing checkout.
REPO_URL = os.environ.get('CLOUDCOST_REPO_URL', '')  # e.g. https://github.com/<org>/CloudCost-IQ.git
REPO_REF = os.environ.get('CLOUDCOST_REPO_REF', 'main')  # branch/tag/commit for reproducibility
PROJECT_ROOT = Path(os.environ.get('CLOUDCOST_PROJECT_ROOT', '/content/CloudCost-IQ')).resolve()

if REPO_URL:
    if not PROJECT_ROOT.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'fetch', '--all', '--tags'])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'checkout', REPO_REF])
    commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
    print('Checked out commit:', commit)
else:
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT = Path.cwd().resolve()
    print('Using existing checkout:', PROJECT_ROOT)

# Install project ML dependencies from the checked-out code when available
for req in (PROJECT_ROOT / 'requirements-ml.txt', PROJECT_ROOT / 'requirements.txt'):
    if req.exists():
        print('Installing', req.name)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)
print('On path:', str(PROJECT_ROOT) in sys.path)


## Section 3 — Load the billing dataset

In [ ]:
# Choose: 'drive' (normalized Parquet path on Drive), 'upload' (manual upload), or 'synthetic_demo' (DEMO ONLY)
DATA_SOURCE = os.environ.get('CLOUDCOST_DATA_SOURCE', 'drive')  # drive | upload | synthetic_demo
CURRENCY = os.environ.get('CLOUDCOST_CURRENCY', 'USD').upper()
COST_BASIS = os.environ.get('CLOUDCOST_COST_BASIS', 'billed')
RECORDS_PATH = None

if DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    default_path = '/content/drive/MyDrive/cloudcost-iq/normalized_billing.parquet'
    RECORDS_PATH = Path(os.environ.get('CLOUDCOST_DATA_PATH', default_path))
    assert RECORDS_PATH.exists(), f'Set CLOUDCOST_DATA_PATH to a normalized Parquet file/dir. Missing: {RECORDS_PATH}'
elif DATA_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()  # upload .parquet or .csv (normalized schema expected for parquet)
    names = list(uploaded.keys())
    assert names, 'No file uploaded'
    RECORDS_PATH = Path('/content') / names[0]
    print('Uploaded:', RECORDS_PATH)
elif DATA_SOURCE == 'synthetic_demo':
    # DEMONSTRATION ONLY — not real billing data, not for production accuracy claims
    print('WARNING: generating SYNTHETIC DEMO billing data (not real). For pipeline demonstration only.')
    import importlib, scripts.create_demo_billing as demo_mod
    demo_out = PROJECT_ROOT / 'data' / 'demo' / 'demo_synthetic_normalized.parquet'
    demo_out.parent.mkdir(parents=True, exist_ok=True)
    # build and save via module logic
    end = pd.Timestamp.today().normalize().date()
    frame_demo = demo_mod.build_frame(days=420, end=end, seed=SEED)
    from app.utils.files import write_parquet_atomic
    write_parquet_atomic(frame_demo, demo_out)
    (demo_out.parent / 'demo_manifest.json').write_text(__import__('json').dumps({
        'synthetic': True,
        'warning': 'SYNTHETIC DEMONSTRATION DATA - not real billing, not for production accuracy claims.',
        'rows': int(len(frame_demo)), 'currency': CURRENCY, 'cost_basis': COST_BASIS
    }, indent=2), encoding='utf-8')
    RECORDS_PATH = demo_out
    print('Synthetic demo data written to:', RECORDS_PATH)
else:
    raise ValueError('DATA_SOURCE must be drive, upload, or synthetic_demo')

print('Records path:', RECORDS_PATH)
print('Cohort:', CURRENCY, COST_BASIS)


## Section 4 — Dataset validation and eligibility

In [ ]:
import json
import pandas as pd
from app.ml.pipeline import load_normalized_records, prepare_training_panel

records = load_normalized_records(RECORDS_PATH)
print('Normalized rows:', f'{len(records):,}')
if 'raw_fields_json' in records.columns:
    sample_raw = str(records['raw_fields_json'].dropna().astype(str).head(1).tolist())
    if 'synthetic' in sample_raw.lower():
        print('*** SYNTHETIC/DEMO DATA DETECTED — results are for demonstration only, not production accuracy. ***')
display(records.head(3))

prepared = prepare_training_panel(records, currency=CURRENCY, cost_basis=COST_BASIS)
quality = prepared.quality
print(json.dumps({k: quality.get(k) for k in (
    'currency','cost_basis','series_count','eligible_series_count',
    'history_start','history_end','history_days','eligible_records','observed_records',
    'missing_periods','zero_cost_days','sufficient','reason','dataset_fingerprint'
)}, indent=2, default=str))

if not prepared.is_sufficient:
    print('Dataset does not meet minimum training eligibility:', prepared.insufficient_reason())
    print('You may still inspect the panel, but a promotable production model requires eligible real data.')


## Section 5 — Cost aggregation (daily panel)

In [ ]:
panel = prepared.eligible_panel if not prepared.eligible_panel.empty else prepared.panel
if panel.empty:
    raise RuntimeError('No daily panel could be built for the selected cohort.')
observed = panel.loc[panel['observed']] if 'observed' in panel.columns else panel
print('Panel rows:', f'{len(panel):,}', '| observed:', f'{len(observed):,}')
print('Series:', panel['series_id'].nunique() if 'series_id' in panel.columns else 'n/a')
print('Date range:', pd.to_datetime(panel['date']).min().date(), 'to', pd.to_datetime(panel['date']).max().date())
if 'observed' in panel.columns:
    print('Missing days kept as missing (not zero):', int((~panel['observed']).sum()))
daily = observed.groupby('date', as_index=False)['cost'].sum().sort_values('date')
display(daily.tail())


## Section 6 — Exploratory data analysis

In [ ]:
import plotly.express as px

daily = observed.groupby('date', as_index=False)['cost'].sum().sort_values('date')
fig = px.line(daily, x='date', y='cost', title='Daily observed cost (eligible cohort)')
fig.show()

if 'provider' in observed.columns:
    by_provider = observed.groupby('provider', as_index=False)['cost'].sum().sort_values('cost', ascending=False)
    px.bar(by_provider, x='provider', y='cost', title='Observed cost by provider').show()

if {'provider','service_name'}.issubset(observed.columns):
    by_service = (observed.groupby(['provider','service_name'], as_index=False)['cost'].sum()
                  .sort_values('cost', ascending=False).head(15))
    px.bar(by_service, x='cost', y='service_name', color='provider', orientation='h', title='Top services').show()

if {'observed','provider','service_name'}.issubset(panel.columns):
    coverage = (panel.groupby(['provider','service_name'])['observed'].mean().sort_values().head(10) * 100).round(1)
    print('Lowest observed coverage series (%):')
    display(coverage.rename('observed_%'))


## Section 7 — Feature engineering (leakage-safe)

In [ ]:
from app.ml.pipeline import build_supervised

assert prepared.is_sufficient, 'Cannot build a valid training set: ' + (prepared.insufficient_reason() or 'insufficient data')
supervised = build_supervised(panel)
print('Supervised rows:', f'{len(supervised.frame):,}')
print('Features:', len(supervised.schema.all_features))
print(supervised.schema.all_features)
display(supervised.frame[[c for c in ['series_id','origin_date','target_date','horizon','target'] if c in supervised.frame.columns]].head())
assert (pd.to_datetime(supervised.frame['target_date']) > pd.to_datetime(supervised.frame['origin_date'])).all()
print('Leakage guard: every target date is after its forecast origin.')


## Section 8 — Backtesting dataset (rolling-origin + untouched test)

In [ ]:
from app.ml.pipeline import make_splits

splits = make_splits(supervised)
for message in splits.messages:
    print('-', message)
print('Valid split plan:', splits.is_valid)
for fold in splits.folds:
    print(fold.name, '| train rows', len(fold.train_index), '| validation', fold.validation_start, 'to', fold.validation_end)
if splits.final_test is not None:
    ft = splits.final_test
    print(ft.name, '| train rows', len(ft.train_index), '| test', ft.validation_start, 'to', ft.validation_end)
assert splits.is_valid, 'Chronological split plan is incomplete (need validation folds and untouched final test).'


## Section 9 — Train XGBoost quantile model (P10/P50/P90)

In [ ]:
from app.ml.train import TrainingConfig, train_forecast_model

config = TrainingConfig.from_ml_config(currency=CURRENCY, cost_basis=COST_BASIS)
train_eval = train_forecast_model(
    supervised, splits, dataset_quality=quality, config=config,
    save_bundle=False, track_mlflow=False,
)
print('Selected rounds:', train_eval.rounds)
print('Passes gates (evaluation):', train_eval.passes_gates)
for gate in train_eval.gates:
    print(f"[{'PASS' if gate['passed'] else 'FAIL'}] {gate['name']}: {gate['detail']}")


## Section 10 — Evaluate vs baselines (out-of-sample)

In [ ]:
import json
import pandas as pd
import plotly.express as px

print('Rolling-origin folds:')
fold_cols = [c for c in ['fold','validation_start','validation_end','mae','rmse','wape','interval_coverage','quantile_crossing_rate','rounds'] if c in (train_eval.metrics.get('folds') or [{}])[0]]
display(pd.DataFrame(train_eval.metrics.get('folds', []))[fold_cols] if fold_cols else pd.DataFrame(train_eval.metrics.get('folds', [])))
print('\nFinal test (out-of-sample):')
print(json.dumps(train_eval.metrics.get('final_test', {}), indent=2, default=str))
print('\nBaseline comparison:')
print(json.dumps(train_eval.metrics.get('baseline_comparison', {}), indent=2, default=str))

predictions = train_eval.evaluation_predictions.copy()
predictions['target_date'] = pd.to_datetime(predictions['target_date'])
plot_frame = (predictions.groupby('target_date', as_index=False)
              .agg(actual=('target','sum'), p10=('p10','sum'), p50=('p50','sum'), p90=('p90','sum'))
              .sort_values('target_date'))
px.line(plot_frame, x='target_date', y=['actual','p10','p50','p90'], title='Out-of-sample actual vs P10/P50/P90').show()


## Section 11 — Explainability (SHAP or feature gain)

In [ ]:
import numpy as np
import pandas as pd

features = supervised.features()
try:
    import shap
    sample_n = min(500, len(features))
    sample = features.head(sample_n)
    explainer = shap.TreeExplainer(train_eval.model)
    values = explainer.shap_values(sample)
    matrix = np.asarray(values)
    if matrix.ndim == 3:
        matrix = matrix[:, :, 1] if matrix.shape[2] > 1 else matrix[:, :, 0]
    importance = (pd.DataFrame({'feature': sample.columns, 'mean_abs_shap': np.abs(matrix).mean(axis=0)})
                  .sort_values('mean_abs_shap', ascending=False))
    display(importance.head(20))
    print('SHAP shown for median-quantile contribution where applicable.')
except Exception as exc:
    print('SHAP unavailable (', type(exc).__name__, '): using native gain importance')
    booster = train_eval.model.get_booster()
    gain = booster.get_score(importance_type='gain')
    imp = pd.DataFrame(sorted(gain.items(), key=lambda x: x[1], reverse=True), columns=['feature','gain'])
    display(imp.head(20))
print('Feature importance indicates model usage, not causation.')


## Section 12 — Final candidate training (save bundle, no auto-promotion)

In [ ]:
from app.ml.artifact_manager import ArtifactManager

final_version = os.environ.get('CLOUDCOST_MODEL_VERSION') or ArtifactManager.new_version()
track_mlflow = os.environ.get('CLOUDCOST_MLFLOW', '0').lower() in ('1','true','yes')

final_result = train_forecast_model(
    supervised, splits, dataset_quality=quality, config=config,
    version=final_version, save_bundle=True, track_mlflow=track_mlflow,
)
print('Candidate version:', final_result.version)
print('Bundle path:', final_result.bundle_path)
print('Passes gates:', final_result.passes_gates)
for msg in final_result.messages:
    print('-', msg)
if not final_result.passes_gates:
    print('*** Candidate FAILED promotion gates — keep in candidates/, do NOT promote/serve. ***')


## Section 13 — Model validation (artifact contract + smoke test)

In [ ]:
from app.ml.model_contract import validate_bundle
from app.ml.artifact_manager import ArtifactManager

assert final_result.bundle_path is not None, 'No bundle was saved'
validation = validate_bundle(final_result.bundle_path)
for check in validation.checks:
    print(f"[{'PASS' if check.passed else 'FAIL'}] {check.name}: {check.detail}")
for err in validation.errors:
    print('ERROR:', err)
print('Validation passed:', validation.passed)

manager = ArtifactManager()
bundle = manager.load_bundle(final_result.version, stage='candidates') if validation.passed else None
if bundle:
    print('Loaded candidate bundle:', bundle.version, 'stage:', bundle.stage)
    print('Quantiles:', bundle.model_config.quantile_alpha, 'Horizon:', bundle.model_config.forecast_horizon_days)
else:
    print('Bundle not loadable as a validated candidate; do not promote.')


## Section 14 — Save model artifacts (inspect bundle)

In [ ]:
from pathlib import Path
bundle_dir = Path(final_result.bundle_path)
print('Bundle contents:')
for p in sorted(bundle_dir.iterdir()):
    size = p.stat().st_size if p.is_file() else 0
    print(f' - {p.name:35s} {size:>10,} bytes')
print('\nKeep this bundle versioned. Promotion to production must be explicit and only after all gates pass.')


## Section 15 — Transfer artifacts to Google Drive / download

In [ ]:
import shutil
from pathlib import Path

bundle_dir = Path(final_result.bundle_path)
copy_to_drive = os.environ.get('CLOUDCOST_COPY_TO_DRIVE', '1').lower() in ('1','true','yes')
if copy_to_drive:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        dest_root = Path(os.environ.get('CLOUDCOST_DRIVE_MODELS_DIR', '/content/drive/MyDrive/cloudcost-iq/models'))
        dest_root.mkdir(parents=True, exist_ok=True)
        dest = dest_root / bundle_dir.name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(bundle_dir, dest)
        print('Copied bundle to Drive:', dest)
    except Exception as exc:
        print('Drive copy skipped/failed:', type(exc).__name__, exc)

# Zip for easy download
zip_path = Path('/content') / f'{bundle_dir.name}.zip'
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', bundle_dir.parent, bundle_dir.name)
print('Zip ready for download:', zip_path)

print('\nTo integrate locally:')
print('1) Place folder under artifacts/models/candidates/<version>')
print('2) Run: python scripts/validate_model.py --version', bundle_dir.name)
if validation.passed and final_result.passes_gates:
    print('3) If passed and you approve: python scripts/validate_model.py --version', bundle_dir.name, '--promote')
else:
    print('3) Do NOT promote: validation/gates did not fully pass.')


## Reproducibility & safety notes

- Record **Git commit/ref**, **package versions** (Section 1), **dataset fingerprint/cohort/date range** (Section 4), and **model version** (Section 12) with any reported metric.
- Final test window is **never** used for hyperparameter/round selection.
- Baselines (trailing 7-day average, seasonal-naive weekly) are benchmarks only.
- Missing days remain missing (never zero-filled). One currency + one cost basis per artifact.
- A candidate that fails gates/validation **must not** be promoted or served. The app only serves a validated production bundle.
- Synthetic demo runs are for pipeline demonstration only — **not** evidence of production accuracy.
